# Plain Arnoldi (no Chebyshev filter) on the anb-compute GPU cluster

`ArnoldiLindblad` is the Chebyshev-free counterpart of `ChebAr`: same target
(the slowest decaying eigenvalue of the one-block propagator), same
interaction-frame hooks, but no spectral filter. It is the reference check on
`ChebAr`'s filtered answer, so this notebook mirrors `tests/test_cheb_ar.ipynb`
cell for cell and the two are directly comparable.

Adapted from the pre-cluster `arnoldi_no_cheb/test_ar_no_cheb*.ipynb`, which
assumed a local GPU (`CUDA_VISIBLE_DEVICES=0`) and did the numerics inline in
the kernel. There is no local GPU any more: the work runs on the anb-compute Ray
cluster, which takes **one picklable function** and gives back **its return
value**, so the whole pipeline lives in `run_arnoldi` below and only plotting
happens here.

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`).
   Those versions must match the cluster; a 3.14 kernel cannot talk to it.

2. Run all cells.

The first cell mints the cluster token itself, so the kernel does **not** need to
have been started from a shell with `RAY_AUTH_TOKEN` exported. It does need to
run before anything imports `ray`, which in a fresh kernel it does — if you have
already imported ray, restart the kernel.

The first `acr.run` of a session also needs GCS read credentials to fetch its
result, which may open an interactive Microsoft device-code login (a URL and a
code printed below the cell). That is expected, and cached afterwards.

Unfiltered Arnoldi needs many more iterations than the filtered version for the
same accuracy — that is the whole point of the comparison — so expect this to be
**slower than `test_cheb_ar.ipynb`** at equal quality.

In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel. If
# you have already run anything that imports ray, restart the kernel first.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

# Mint the cluster token from Vault here, so the notebook also works from a
# kernel that was not launched from a shell with RAY_AUTH_TOKEN exported (which
# is the normal case in VS Code). `vault login` is interactive, so it stays a
# manual, once-a-week step.
if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import ray

from anb_compute import ray as acr

print("python", sys.version.split()[0], "| ray", ray.__version__)

import pathlib


def _find_repo():
    """Locate the Cheb_Ar checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "cheb_ar").is_dir():
            return base
        if (base / "Cheb_Ar" / "src" / "cheb_ar").is_dir():
            return base / "Cheb_Ar"
    raise RuntimeError(f"no Cheb_Ar checkout found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
CHEB_AR = str(REPO / "src" / "cheb_ar")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (CHEB_AR, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[CHEB_AR, DYNAMIQS])

In [ ]:
def run_arnoldi(
    n_a=25,
    n_b=11,
    alpha_sq=6,
    epsilon_p=0.1,
    n_periods=1,
    m_arnoldi=120*6,
    seed=0,
    stride=1,
):
    """Plain (unfiltered) Arnoldi for one parameter point, on one GPU worker.

    Returns plain numpy/python objects only: the client has neither jax nor
    dynamiqs installed, so nothing jax-flavoured may cross back. The Wigner grid
    is therefore computed here too, and the client only plots it.

    Physics constants come from `cheb_ar.models.ats` (the single source of
    truth); only the knobs that actually vary are arguments. The pre-cluster
    notebook re-declared w_a / E_J / phi_a / phi_b by hand, which is exactly the
    duplication that module is meant to prevent.
    """
    import time

    import numpy as np

    import dynamiqs as dq
    from cheb_ar import ArnoldiLindblad
    from cheb_ar.models.ats import build_ats_hamiltonian_interaction

    def py(v):
        """0-d jax/numpy scalar -> plain python float/complex."""
        a = np.asarray(v)
        return complex(a) if np.iscomplexobj(a) else float(a)

    t0 = time.time()
    dims = (n_a, n_b)
    N = n_a * n_b

    # --- interaction frame: mesolve_fast + output_phase + frame transform V ---
    (
        H_I,
        jump_ops_I,
        jump_ops_LdL_I,
        output_phase,
        V,
        T_block,
        params,
    ) = build_ats_hamiltonian_interaction(
        n_a=n_a,
        n_b=n_b,
        alpha_sq=alpha_sq,
        epsilon_p=epsilon_p,
        n_periods=n_periods,
    )

    solver = ArnoldiLindblad(
        H_I,
        jump_ops_I,
        T_block,
        jump_ops_LdL=jump_ops_LdL_I,
        dims=dims,
        output_phase=output_phase,
    )
    t_build = time.time() - t0

    # --- unfiltered Arnoldi; mu_ref=1 tracks the slowest decaying mode ---
    x0 = solver.make_x0(seed=seed)
    Q, H, steps, mu_list = solver.arnoldi(x0, m_arnoldi, stride=stride)
    t_arnoldi = time.time() - t0

    # --- rate + residual on the true propagator ---
    # `steps[-1]` is the step count the factorization actually reached, which is
    # where mu_list[-1] came from. Passing a smaller m would silently discard
    # Krylov vectors and give a worse eigenvector than mu_list[-1] suggests.
    m_reached = int(steps[-1])
    rate_bf = solver.rate_from_mu(mu_list[-1])
    x_ritz, _ = solver.ritz_vector(Q, H, m_reached, target=mu_list[-1])
    res = solver.residual_check(x_ritz)

    # --- back to the lab frame, single-mode operators and Wigners ---
    # Use `solver.unvec`, not a bare reshape(N, N): the convention is
    # column-major, so a plain reshape hands back the transpose (the complex
    # conjugate, for Hermitian input).
    V = np.asarray(V)
    rho_lab = V @ np.asarray(solver.unvec(x_ritz)) @ V.conj().T
    rho_a = dq.ptrace(dq.asqarray(rho_lab, dims=dims), 0)
    xvec_a, yvec_a, W_a = dq.wigner(rho_a)
    rho_b = dq.ptrace(dq.asqarray(rho_lab, dims=dims), 1)
    xvec_b, yvec_b, W_b = dq.wigner(rho_b)

    # --- per-level Fock weight, for the truncation check (see test_cheb_ar) ---
    D = np.einsum("ij,ij->i", rho_lab, rho_lab.conj()).real.reshape(n_a, n_b)

    # Q is (m+1, N^2) complex -- hundreds of MB. It must NOT cross the wire;
    # only H (tiny) and the Ritz vector are returned.
    H = np.asarray(H)

    import jax

    return {
        "rate_bf": py(rate_bf),
        "mu": py(mu_list[-1]),
        "mu_RQ": py(res["mu_RQ"]),
        "rate_RQ": py(res["rate_RQ"]),
        "res_rel": py(res["res_rel"]),
        "steps": np.asarray(steps),
        "mu_list": np.asarray(mu_list),
        # H is real up to integrator noise, so this is a free noise-floor estimate
        "H_imag_max": float(np.abs(H.imag).max()),
        "H": H,
        "x_ritz": np.asarray(x_ritz),
        "rho_a": np.asarray(dq.to_jax(rho_a)),
        "rho_b": np.asarray(dq.to_jax(rho_b)),
        "fock_weight_a": D.sum(axis=1) / D.sum(),
        "fock_weight_b": D.sum(axis=0) / D.sum(),
        "wigner_a": (np.asarray(xvec_a), np.asarray(yvec_a), np.asarray(W_a)),
        "wigner_b": (np.asarray(xvec_b), np.asarray(yvec_b), np.asarray(W_b)),
        "params": {k: py(v) for k, v in params.items()},
        "settings": {
            "n_a": n_a,
            "n_b": n_b,
            "alpha_sq": alpha_sq,
            "epsilon_p": epsilon_p,
            "n_periods": n_periods,
            "m_arnoldi": m_arnoldi,
            "m_reached": m_reached,
            "seed": seed,
            "stride": stride,
        },
        "timings_s": {
            "build": t_build,
            "arnoldi": t_arnoldi - t_build,
            "total": time.time() - t0,
        },
        "device": str(jax.devices()[0]),
    }

In [ ]:
import time

# num_gpus > 0 routes to the gpu pool (on-demand only) and runs `run_arnoldi` as
# a task on a worker rather than inline in the driver. Logs print when the job
# ends; follow a long one with `ray job logs --follow <job_id>`.
t0 = time.time()
result = acr.run(
    run_arnoldi,
    num_gpus=1,
    num_cpus=8,
    runtime_env=RUNTIME_ENV,
    poll_seconds=5,
)
print(f"\nwall clock, incl. job startup: {time.time() - t0:.1f} s")
print("device :", result["device"])
print("timings:", {k: round(v, 1) for k, v in result["timings_s"].items()})

In [ ]:
s = result["settings"]
print("rate_bf    :", result["rate_bf"])
print("mu         :", result["mu"])
print("mu_RQ      :", result["mu_RQ"])
print("rate_RQ    :", result["rate_RQ"])
print("res_rel    :", result["res_rel"])
print("m reached  :", s["m_reached"], f"(requested {s['m_arnoldi']})")
# H is real up to integrator noise; this is the noise floor res_rel is fighting
print("max|Im H|  :", result["H_imag_max"])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot against `steps`, not range(m_arnoldi): with stride > 1 they differ, and
# `steps` is what mu_list is indexed by.
steps, mu_list = result["steps"], result["mu_list"]

fig, ax = plt.subplots(figsize=(5.4, 3.8), constrained_layout=True)
ax.plot(steps, np.abs(1 - mu_list), ".-", ms=4, lw=0.8)
ax.axhline(result["res_rel"], ls="--", lw=0.8, c="k", label="res_rel")
ax.axhline(result["H_imag_max"], ls=":", lw=0.8, c="0.5", label="max|Im H|")
ax.set_yscale("log")
ax.set_xlabel("Arnoldi iteration")
ax.set_ylabel(r"$|1-\mu|$")
ax.set_title(
    rf"unfiltered Arnoldi, $|\alpha|^2 = {result['settings']['alpha_sq']}$"
)
ax.legend(fontsize=8)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

xvec, yvec, W = result["wigner_a"]

# Same convention as dq.plot.wigner: W is indexed [x, y], so transpose for
# pcolormesh, and clip to +-2/pi.
vmax = 2 / np.pi
fig, ax = plt.subplots(figsize=(5.0, 4.2), constrained_layout=True)
im = ax.pcolormesh(
    xvec, yvec, np.clip(W.T, -vmax, vmax),
    cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto",
)
ax.set_aspect("equal")
ax.set_xlabel(r"$\mathrm{Re}(\alpha)$")
ax.set_ylabel(r"$\mathrm{Im}(\alpha)$")
ax.set_title(rf"memory mode, $|\alpha|^2 = {result['settings']['alpha_sq']}$")
fig.colorbar(im, ax=ax, ticks=[-vmax, 0, vmax], label="W")
plt.show()

In [ ]:
xvec, yvec, W = result["wigner_b"]

# Same convention as dq.plot.wigner: W is indexed [x, y], so transpose for
# pcolormesh, and clip to +-2/pi.
vmax = 2 / np.pi
fig, ax = plt.subplots(figsize=(5.0, 4.2), constrained_layout=True)
im = ax.pcolormesh(
    xvec, yvec, np.clip(W.T, -vmax, vmax),
    cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto",
)
ax.set_aspect("equal")
ax.set_xlabel(r"$\mathrm{Re}(\beta)$")
ax.set_ylabel(r"$\mathrm{Im}(\beta)$")
ax.set_title(rf"buffer mode, $|\alpha|^2 = {result['settings']['alpha_sq']}$")
fig.colorbar(im, ax=ax, ticks=[-vmax, 0, vmax], label="W")
plt.show()